In [35]:
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [36]:
df = pd.read_excel('/Users/MAC/PycharmProjects/диплом от 15.11/отфильтрованая таблица 19_04_26 с Killip.xlsx')

In [37]:
#формируем датафрейм с признаками (предикторами)
#непрерывные
cont_p_value_less_05 =[
'GRACE(Рассчет)',
'Систолическое АД(b)',
'Apache II',
'Диастолического АД(b)',
'Нейтрофилы (относительное значение)',
'Лейкоциты(a)',
'Нейтрофилы (абсолютное значение)',
'Нейтрофилы (абсолютное значение)(a)',
'Лимфоциты (относительное значение)',
'SpO2',
'Базофилы (относительное значение)',
'Эозинофилы (относительное значение)',
'Лейкоциты',
'СКФ',
'Возраст',
'eGFR',
'Эозинофилы (абсолютное значение)',
'ЧСС (b)',
'Глюкоза(a)',
'ФВ ЛЖ',
'СДЛА',
'Гематокрит(a)',
'Мочевина(b)',
'BLR (базофилы абс/лимфоциты абс)',
'Глюкоза в мг/дл',
'АЛАТ(a)',
'ELR (эозинофилы абс/лимфоциты абс',
'Лимфоциты (абсолютное значение)',
'МНО(a)',
'ПТИ(a)',
'ENR (эозин абс/нейтрофилы абс)',
'Hb(a)',
'Гематокрит',
'Эр(a)',
'Гемоглобин',
'Базофилы (абсолютное значение)',
'Эритроциты',
'АСАТ(a)',
'Креатинин(a)',
'Креатинин',
'Триглицериды(a)',
'Средний объем тромбоцита (MPV)',
'Вес',
'НПВ диаметр',
'Моноциты (относительное значение)',
'Продолжительность операции',
'КФК(a)',
'SII (тромбоциты*нейтрофилы абс / лимфоциты абс)',
'RLR (RDW/ lymph abs)',
'Моноциты (абсолютное значение)',
'ЗСЛЖ',
'Рост',
'Распределение эритроцитов по объему (RDW-CV)',
'КСР ЛЖ',
'АПТВ(a)',
'PLR (тромбоциты/лимфоциты абс) (61-239)',
'La1',
'La2',
'Холестерин общий(a)',
'МЖП',


'Killip',
'killip',
'Класс ОСН по Killip']
#поменяла 14/03


#дихотомические
dih_p_value_less_05 =[
'GRACE (Высокий риск)',
'Отек легких',
'GRACE(Общее)>156',
'TIMI (Летальность) (Высокий риск)',
'GRACE(Общее)>140',
'Отек легких(b)',
'CADILLAC (Высокий риск)',
'интегрилин|эптифибатид|коромакс|агграстат',
'GRACE (Низкий риск)',
'РЕКОРД (Низкий риск)',
'РЕКОРД (Высокий риск)',
'Отек легких(a)',
'ФП b (после чкв)',
'А-В блокаДа',
'CADILLAC (Низкий риск)',
'PAMI (Высокий риск)',
'Интегреллин',
'Поражение ствола',
'GRACE (Средний риск)',
'Тирофибан',
'Инфаркт миокарда со стентированием в анамнезе',
'ФВ<30%',
'TIMI (Летальность) (Низкий риск)',
'Стенокардия в диагнозе при поступлении',
'ХОБЛ',
'Тромболизис',
'Общий анализ крови_экспресс раньше операции',
'СД',
'ФП a (в анамнезе)',
'ХБП',
'PAMI (Низкий риск)',
'ПИКС в диагнозе при поступлении',
'Левосимендан',
'Медицинская помощь оказана за первые 4 часа',
'PAMI (Средний риск)',
'Гипертоническая болезнь',
'TIMI категория',
] #поменяла 14/03

#категориальные признаки
cat_p_value_less_05 =[
'Количество пораженных сосудов(Syntax)',
'Количество пораженных сосудов(Значимость)',
'CADILLAC категория',
'РЕКОРД категория',
'PAMI категория',
'TIMI Летальность категория',
'Вид операции(ИБ)(Новый)',
'Исход заболевания',
'Инфаркт-зависимая артерия(Огригированная)',
'KDIGO',
'Baun',
'MKB категория',
'APACHE2 выписка категория',
'APACHE2 1сутки категория',
'APACHE2 3дня категория',
'ГБ стадия категория'
]


all_predictors = cont_p_value_less_05 + dih_p_value_less_05 + cat_p_value_less_05 + ['КШ развился в реанимации']
predict_df = df[all_predictors]



y = predict_df['КШ развился в реанимации']
X = predict_df.drop('КШ развился в реанимации', axis=1)



#сразу закодируем признаки (категориальные)
for col in cat_p_value_less_05:
    if col in X.columns:
        # Преобразуем все значения в строки и заменяем NaN на специальную метку
        X[col] = X[col].astype(str)

transformers = [
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_p_value_less_05),
    ('cont', StandardScaler(), cont_p_value_less_05)
]

preprocessor = ColumnTransformer(transformers=transformers, remainder='passthrough')

#применяем ColumnTransformer
X_encoded = preprocessor.fit_transform(X)

#cоздаем имена столбцов для закодированных категориальных признаков
encoded_feature_names_cat = preprocessor.transformers_[0][1].get_feature_names_out(cat_p_value_less_05)


#cоздаем DataFrame с закодированными признаками
#получаем индексы столбцов, которые не были закодированы
remainder_indices = [i for i, col in enumerate(X.columns) if col not in cat_p_value_less_05]

#получаем имена столбцов, которые не были закодированы
remainder_feature_names = [X.columns[i] for i in remainder_indices]

#объединяем названия всех столбцов
all_column_names = list(encoded_feature_names_cat) +remainder_feature_names
#cоздаем DataFrame с закодированными признаками
X_encoded_df = pd.DataFrame(X_encoded, columns=all_column_names)
X_y_encoded_df = pd.concat([X_encoded_df, y], axis=1)
X_y_encoded_df.to_excel('датафрейм с предикторами 19_04_26 с Killip.xlsx')

In [38]:
pd.set_option('display.max_columns', None)
X_y_encoded_df.head(10)

,Количество пораженных сосудов(Syntax)_0,Количество пораженных сосудов(Syntax)_1,Количество пораженных сосудов(Syntax)_2,Количество пораженных сосудов(Syntax)_3,Количество пораженных сосудов(Syntax)_4,Количество пораженных сосудов(Значимость)_0,Количество пораженных сосудов(Значимость)_1,Количество пораженных сосудов(Значимость)_2,Количество пораженных сосудов(Значимость)_3,CADILLAC категория_Высокий уровень,CADILLAC категория_Неизвестно,CADILLAC категория_Низкий уровень,CADILLAC категория_Средний уровень,РЕКОРД категория_Высокий риск,РЕКОРД категория_Неизвестно,РЕКОРД категория_Низкий риск,РЕКОРД категория_Очень высокий,РЕКОРД категория_Умеренный риск,PAMI категория_Высокий риск,PAMI категория_Неизвестно,PAMI категория_Низкий риск,PAMI категория_Средний риск,TIMI Летальность категория_Высокий риск,TIMI Летальность категория_Неизвестно,TIMI Летальность категория_Низкий риск,TIMI Летальность категория_Очень высокий риск,TIMI Летальность категория_Умеренный риск,Вид операции(ИБ)(Новый)_-,Вид операции(ИБ)(Новый)_nan,Вид операции(ИБ)(Новый)_Аспирация тромботических масс,Вид операции(ИБ)(Новый)_Имплантация,Вид операции(ИБ)(Новый)_Лапароскопическая аппендектомия,Вид операции(ИБ)(Новый)_Реваскуляризация,Вид операции(ИБ)(Новый)_Реканализация,Вид операции(ИБ)(Новый)_ТЛБАП,Вид операции(ИБ)(Новый)_ТЛБАП и стентирование ПМЖВ,Вид операции(ИБ)(Новый)_ТЛБАП со стентированием ОВ,Вид операции(ИБ)(Новый)_ТЛБАП со стентированием ОВи ПМЖВ,"Вид операции(ИБ)(Новый)_ТЛБАП со стентированием ПКА, ВТК1",Вид операции(ИБ)(Новый)_ТЛБАП со стентированием ПМЖВ,Вид операции(ИБ)(Новый)_Тромбаспирация,Вид операции(ИБ)(Новый)_ЧКА,Вид операции(ИБ)(Новый)_Эндартерэктомия,Вид операции(ИБ)(Новый)_чка,Исход заболевания_без перемен,Исход заболевания_выздоровление,Исход заболевания_неоконченный случай с улучшением,Исход заболевания_переведен в другой стационар,Исход заболевания_переведен в другой стационар с улучшением,Исход заболевания_с выздоровлением,Исход заболевания_с улучшением,Исход заболевания_самоуход,Исход заболевания_умер,Инфаркт-зависимая артерия(Огригированная)_nan,Инфаркт-зависимая артерия(Огригированная)_Бассейн ОВ,Инфаркт-зависимая артерия(Огригированная)_Бассейн ПКА,Инфаркт-зависимая артерия(Огригированная)_Бассейн левой КА,Инфаркт-зависимая артерия(Огригированная)_Ствол,KDIGO_0,KDIGO_1,KDIGO_2,KDIGO_3,Baun_0,Baun_1,Baun_2,Baun_3,Baun_4,Baun_5,Baun_6,MKB категория_nan,"MKB категория_КЛАСС III БОЛЕЗНИ КРОВИ, КРОВЕТВОРНЫХ ОРГАНОВ И ОТДЕЛЬНЫЕ НАРУШЕНИЯ, ВОВЛЕКАЮЩИЕ ИММУННЫЙ МЕХАНИЗМ (D50-D89)","MKB категория_КЛАСС IV БОЛЕЗНИ ЭНДОКРИННОЙ СИСТЕМЫ, РАССТРОЙСТВА ПИТАНИЯ И НАРУШЕНИЯ ОБМЕНА ВЕЩЕСТВ (E00-E90)",MKB категория_КЛАСС IX БОЛЕЗНИ СИСТЕМЫ КРОВООБРАЩЕНИЯ (I00-I99),MKB категория_КЛАСС VI БОЛЕЗНИ НЕРВНОЙ СИСТЕМЫ (G00-G99),MKB категория_КЛАСС X БОЛЕЗНИ ОРГАНОВ ДЫХАНИЯ (J00-J99),MKB категория_КЛАСС XI БОЛЕЗНИ ОРГАНОВ ПИЩЕВАРЕНИЯ (K00-K93),MKB категория_КЛАСС XIII БОЛЕЗНИ КОСТНО-МЫШЕЧНОЙ СИСТЕМЫ И СОЕДИНИТЕЛЬНОЙ ТКАНИ (M00-M99),MKB категория_КЛАСС XIV БОЛЕЗНИ МОЧЕПОЛОВОЙ СИСТЕМЫ (N00-N99),"MKB категория_КЛАСС XIX ТРАВМЫ, ОТРАВЛЕНИЯ И НЕКОТОРЫЕ ДРУГИЕ ПОСЛЕДСТВИЯ ВОЗДЕЙСТВИЯ ВНЕШНИХ ПРИЧИН (S00-T98)",APACHE2 выписка категория_Высокий риск (17-24),APACHE2 выписка категория_Критический риск (25+),APACHE2 выписка категория_Низкий риск (0-10),APACHE2 выписка категория_Умеренный риск (11-16),APACHE2 1сутки категория_Высокий риск (17-24),APACHE2 1сутки категория_Критический риск (25+),APACHE2 1сутки категория_Неизвестно,APACHE2 1сутки категория_Низкий риск (0-10),APACHE2 1сутки категория_Умеренный риск (11-16),APACHE2 3дня категория_Высокий риск (17-24),APACHE2 3дня категория_Критический риск (25+),APACHE2 3дня категория_Неизвестно,APACHE2 3дня категория_Низкий риск (0-10),APACHE2 3дня категория_Умеренный риск (11-16),ГБ стадия категория_Высокий уровень,ГБ стадия категория_Неизвестно,ГБ стадия категория_Низкий уровень,ГБ стадия категория_Средний уровень,GRACE(Рассчет),Систолическое АД(b),Apache II,Диастолического АД(b),Нейтрофилы (относительное значение),Лейкоциты

In [39]:
X_y_encoded_df['КШ развился в реанимации'].value_counts()

КШ развился в реанимации
0    5682
1     200
Name: count, dtype: int64

In [40]:
X_y_encoded_df.shape

(5882, 198)